# Zmiany semantyczne słów

<img src="https://live.staticflickr.com/65535/54942563983_f3baea0eee_c.jpg" alt="Embedded Photo" width="776">

## Wstęp

Znaczenia słów w języku naturalnym zmieniają się wraz z czasem — jedne ewoluują stopniowo, inne przechodzą gwałtowne przesunięcia semantyczne. Dzięki dawnym korpusom tekstowym możliwe jest trenowanie modeli wektorowych, które uchwytują te historyczne różnice.

W tym zadaniu będziesz analizować **pretrenowane embeddingi (word2vec)** z dwóch różnych epok:
- **1900** — model wytrenowany wyłącznie na danych z początku XX wieku  
- **1990** — model wytrenowany na danych bliskich współczesności  

Te dwa modele zostały wytrenowane **całkowicie niezależnie**. Twoim zadaniem będzie wykorzystać te reprezentacje, aby zbudować klasyfikator określający, czy **znaczenie danego słowa uległo istotnej zmianie między rokiem 1900 a 1990**.

## Zadanie

Zbudujesz **klasyfikator binarny**, który dla zadanego słowa zwróci etykietę:

- **0 — słowo stabilne semantycznie**
- **1 — słowo, którego znaczenie się zmieniło**

## Dane

- `train.csv` — zbiór treningowy (słowo + etykieta)
- `valid.csv` — zbiór walidacyjny do lokalnego testowania
- `1900-vocab.pkl` oraz `1900-w.npy` — słownik i macierz embeddingów z 1900 roku  
- `1990-vocab.pkl` oraz `1990-w.npy` — analogiczny zestaw dla roku 1990  

## Kryterium oceny

Do oceny Twojego rozwiązania wykorzystujemy metrykę **Balanced Accuracy**, czyli średnią dokładności klasyfikacji dla klasy pozytywnej i dla klasy negatywnej. Innymi słowy:

$$ \text{Balanced Accuracy} = \frac{1}{2}(\text{TPR} + \text{TNR}) $$

czyli średnią z czułości (TPR) i specyficzności (TNR).  
Jest to metryka odporna na niezbalansowanie zbiorów danych.

Zwracasz **twarde etykiety (0/1)**.

W notebooku znajduje się funkcja `evaluate_algorithm`, dzięki której przetestujesz swój model na `valid.csv`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Wynik będzie skalowany liniowo w zależności od wartości Balanced Accuracy:

- **Balanced Accuracy ≤ 0.7**: 0 punktów.
- **Balanced Accuracy ≥ 0.87**: 100 punktów.
- **Wartości pomiędzy 0.7 a 0.87**: skalowane liniowo.

Wzór na wynik:  
$$
\text{Punkty} = 
\begin{cases} 
0 & \text{dla } \text{Balanced Accuracy} \leq 0.7 \\
100 \times \frac{\text{Balanced Accuracy} - 0.7}{0.87 - 0.7} & \text{dla } 0.7 < \text{Balanced Accuracy} < 0.87 \\
100 & \text{dla } \text{Balanced Accuracy} \geq 0.87
\end{cases}
$$

## Ograniczenia

Twój notebook będzie uruchamiany na Platformie Konkursowej:

- **bez dostępu do internetu**
- bez dostępu do GPU - **CPU only**
- Limit czasu wykonania notebooka i ewaluacji na zbiorze testowym: **5 minut**
- Lista dopuszczalnych bibliotek: `numpy`, `pandas`, `scikit-learn`, `matplotlib`, `tqdm`

## Pliki zgłoszeniowe

- Ten notebook uzupełniony o Twoje rozwiązanie:

```python
class SemanticChangeModel:
    def fit(self, train_df):
        ...
    def predict_change(self, words: List[str]) -> List[int in {0,1}]:
        ...
```

## Ewaluacja

Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz, będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie wyżej wspomnianego wzoru, zaokrąglona do liczby całkowitej. Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo, otrzymasz za zadanie 0 punktów.

## Kod Startowy
W tej sekcji inicjalizujemy środowisko poprzez zaimportowanie potrzebnych bibliotek i funkcji. Przygotowany kod ułatwi Tobie efektywne operowanie na danych i budowanie właściwego rozwiązania.

In [2]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

FINAL_EVALUATION_MODE = False  # Podczas sprawdzania ustawimy tę flagę na True.

import os, json, pickle, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from sklearn.metrics import balanced_accuracy_score

np.random.seed(42)
random.seed(42)

DATA_DIR = Path('data')
EMB_DIR  = DATA_DIR / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)


## Pobieranie danych (tylko lokalnie)

In [3]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

# Uwaga: Na platformie oceniającej Internet jest wyłączony.
# Ten blok uruchamia się tylko lokalnie (gdy FINAL_EVALUATION_MODE == False).

GDRIVE_FILES = [
    ('1rUGgDZcpwRZ5sRHGxxEh2f7ZJ0DRVDPL', EMB_DIR / '1900-vocab.pkl'),
    ('1cYXPhghcawbMZ6vU2XyJUq7NOpBIKj5E', EMB_DIR / '1900-w.npy'),
    ('1ApLkBn2ylvLMKNlNtvkMVde6RxnLJolI', EMB_DIR / '1990-vocab.pkl'),
    ('1B3NLInA4Ty3lUaHNQgxtDTKJNtG0t0T1', EMB_DIR / '1990-w.npy'),
    ('1hrOfZOq3BV1K0tWe6HSZG-OiZkGlCiYT', DATA_DIR / 'train.csv'),
    ('1vndyCuDCBP6zLvNkF_YsKHgTQgulTjt_', DATA_DIR / 'valid.csv'),
]

def download_data():
    try:
        import gdown
    except Exception as e:
        raise RuntimeError('Zainstaluj gdown lokalnie: `pip install gdown`') from e

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    EMB_DIR.mkdir(parents=True, exist_ok=True)

    for fid, out_path in GDRIVE_FILES:
        if out_path.exists():
            print(f'Pominięto pobieranie — plik już istnieje: {out_path.name}')
            continue
        url = f'https://drive.google.com/uc?id={fid}'
        out_path.parent.mkdir(parents=True, exist_ok=True)
        print(f'Pobieranie -> {out_path.name}')
        gdown.download(url, str(out_path), quiet=False)

if not FINAL_EVALUATION_MODE:
    download_data()
    print('Pobieranie zakończone.')
else:
    print('FINAL_EVALUATION_MODE=True — pomijam pobieranie (na platformie dane są dostarczone).')

Pominięto pobieranie — plik już istnieje: 1900-vocab.pkl
Pominięto pobieranie — plik już istnieje: 1900-w.npy
Pominięto pobieranie — plik już istnieje: 1990-vocab.pkl
Pominięto pobieranie — plik już istnieje: 1990-w.npy
Pominięto pobieranie — plik już istnieje: train.csv
Pominięto pobieranie — plik już istnieje: valid.csv
Pobieranie zakończone.


## Ładowanie embeddingów i zbiorów danych

In [4]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

def load_histwords_decade(decade: int, emb_dir: Path):
    vocab_path = emb_dir / f'{decade}-vocab.pkl'
    w_path     = emb_dir / f'{decade}-w.npy'
    with open(vocab_path, 'rb') as f:
        vocab = pickle.load(f)
    W = np.load(w_path)
    # Normalizacja L2
    W = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-12)
    w2i = {w:i for i,w in enumerate(vocab)}
    return vocab, W, w2i

# Wczytanie embeddingów
vocab_1900, W1900, w2i_1900 = load_histwords_decade(1900, EMB_DIR)
vocab_1990, W1990, w2i_1990 = load_histwords_decade(1990, EMB_DIR)

if not FINAL_EVALUATION_MODE:
    print(f'1900: V={len(vocab_1900):,}, dim={W1900.shape[1]}')
    print(f'1990: V={len(vocab_1990):,}, dim={W1990.shape[1]}')

# Wczytanie zbiorów train/valid
train_path = DATA_DIR / 'train.csv'
valid_path = DATA_DIR / 'valid.csv'
assert train_path.exists() and valid_path.exists(), 'Brak train.csv / valid.csv w folderze data/'

train_df = pd.read_csv(train_path)
valid_df = pd.read_csv(valid_path)

# Oczekiwane kolumny: word, label
for c in ['word', 'label']:
    assert c in train_df.columns and c in valid_df.columns, 'Oczekiwane kolumny: word, label'

train_df['word'] = train_df['word'].astype(str).str.lower().str.strip()
valid_df['word'] = valid_df['word'].astype(str).str.lower().str.strip()
train_df['label'] = train_df['label'].astype(int)
valid_df['label'] = valid_df['label'].astype(int)


if not FINAL_EVALUATION_MODE:
    print(train_df.head(5))
    print()
    print(valid_df.head(5))
    print(f'train: {len(train_df)}, valid: {len(valid_df)}')

1900: V=100,000, dim=300
1990: V=100,000, dim=300
        word  label
0     lichen      0
1    imaging      1
2      devil      0
3    prayers      0
4  frankfort      1

        word  label
0  coastline      0
1       yoke      0
2     report      1
3   language      0
4       barn      0
train: 2495, valid: 832


In [5]:
def print_neighbors(word, V, vocab, w2i, label):
    vec = V[w2i[word]]
    sims = V @ vec
    sims[w2i[word]] = -np.inf
    top = np.argsort(-sims)[:10]
    print(f"\nTop 10 sąsiadów w {label}:")
    for i in top:
        print(f"  {vocab[i]}  ({sims[i]:.4f})")


if not FINAL_EVALUATION_MODE:
    print_neighbors("intelligence", W1900, vocab_1900, w2i_1900, "1900")
    print_neighbors("intelligence", W1990, vocab_1990, w2i_1990, "1990")


Top 10 sąsiadów w 1900:
  intellect  (0.4798)
  sagacity  (0.4282)
  discernment  (0.3707)
  understanding  (0.3688)
  tact  (0.3659)
  skill  (0.3643)
  humanity  (0.3641)
  honesty  (0.3598)
  ability  (0.3517)
  sensibility  (0.3512)

Top 10 sąsiadów w 1990:
  iq  (0.3773)
  wechsler  (0.3542)
  cia  (0.3362)
  hinsley  (0.3301)
  artificial  (0.3264)
  afric  (0.3202)
  binet  (0.3191)
  aptitude  (0.3152)
  abwehr  (0.3108)
  abilities  (0.3070)


## Twoje rozwiązanie

In [8]:
#########################  ZMODYFIKUJ TYLKO TĘ KOMÓRKĘ  #########################
# Zaimplementuj swój model jako klasę z metodami:
#   - __init__       : zapisz osadzenia (embeddings) + podstawowe hiperparametry
#   - fit(train_df)  : trenuj na oznaczonych danych
#   - predict_change(words) : zwróć etykiety dla podanej listy słów
#
# Kod ewaluacyjny otrzyma instancję klasy i będzie zakładał jedynie, że posiada
# metodę .predict_change(words).

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

class SemanticChangeModel:
    def __init__(self, W1900, W1990, w2i_1900, w2i_1990):
        self.W1900 = W1900
        self.W1990 = W1990
        self.w2i_1900 = w2i_1900
        self.w2i_1990 = w2i_1990
        self.i2w_1900 = {i: w for w, i in w2i_1900.items()}
        self.i2w_1990 = {i: w for w, i in w2i_1990.items()}
        self.Q = None 
        self.pivots_1900 = None
        self.pivots_1990 = None
        self.clf = None 
        self.scaler = None
        
    def fit(self, train_df):
        print("Start...")
        kf = KFold(n_splits=5, shuffle=True, random_state=42)
        X_unbiased = []
        y_unbiased = []
        train_df = train_df.reset_index(drop=True)
        for train_idx, val_idx in kf.split(train_df):
            fold_train = train_df.iloc[train_idx]
            fold_val = train_df.iloc[val_idx]
            stable_train = fold_train[fold_train['label'] == 0]
            anchors_sup = [w for w in stable_train['word'] if w in self.w2i_1900 and w in self.w2i_1990]
            Q_fold = self._get_rank_stable_Q(anchors_sup)
            anchors_sup.sort(key=lambda w: self.w2i_1900[w])
            fold_pivots = anchors_sup[:1000]
            pivots_1900_fold = np.array([self.W1900[self.w2i_1900[w]] for w in fold_pivots])
            pivots_1990_fold = np.array([self.W1990[self.w2i_1990[w]] for w in fold_pivots])
            X_fold = self._extract_features_std(fold_val['word'].tolist(), Q_fold, pivots_1900_fold, pivots_1990_fold)
            y_fold = fold_val['label'].values
            X_unbiased.append(X_fold)
            y_unbiased.append(y_fold)
        X_train_final = np.vstack(X_unbiased)
        y_train_final = np.concatenate(y_unbiased)
        all_stable = list(set(train_df[train_df['label'] == 0]['word']))
        all_anchors = [w for w in all_stable if w in self.w2i_1900 and w in self.w2i_1990]
        all_stable = list(set(train_df[train_df['label'] == 0]['word']))
        all_anchors = [w for w in all_stable if w in self.w2i_1900 and w in self.w2i_1990]
        all_anchors.sort(key=lambda w: self.w2i_1900[w])
        self.pivots = all_anchors[:1000]
        self.pivots_1900 = np.array([self.W1900[self.w2i_1900[w]] for w in self.pivots])
        self.pivots_1990 = np.array([self.W1990[self.w2i_1990[w]] for w in self.pivots])
        self.Q = self._get_rank_stable_Q(all_anchors)
        print("Obliczono alignment")
        self.scaler = StandardScaler()
        X_scaled = self.scaler.fit_transform(X_train_final)
        from sklearn.ensemble import HistGradientBoostingClassifier
        clf1 = RandomForestClassifier(n_estimators=1000, max_depth=12, class_weight='balanced', random_state=42)
        clf2 = HistGradientBoostingClassifier(learning_rate=0.03, max_iter=1000, max_leaf_nodes=31, random_state=42, class_weight='balanced')
        clf3 = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
        self.clf = VotingClassifier(estimators=[
            ('rf', clf1), ('hgb', clf2), ('lr', clf3)
        ], voting='soft', weights=[4, 4, 1])
        self.clf.fit(X_scaled, y_train_final)
        print("Nauczono za pomocą VotingEnsemble")
        
    def _get_rank_stable_Q(self, anchors):
        if not anchors: return np.eye(self.W1900.shape[1])
        stable_anchors = []
        for w in anchors:
            r1 = np.log(self.w2i_1900[w] + 1)
            r2 = np.log(self.w2i_1990[w] + 1)
            if abs(r1 - r2) < 1.5: 
                stable_anchors.append(w)
        if len(stable_anchors) < 50: stable_anchors = anchors
        idx_1900 = [self.w2i_1900[w] for w in stable_anchors]
        idx_1990 = [self.w2i_1990[w] for w in stable_anchors]
        A = self.W1900[idx_1900]
        B = self.W1990[idx_1990]
        ranks = np.array([idx for idx in idx_1900])
        weights = 1.0 / np.log(ranks + 2)
        weights = weights / np.mean(weights) 
        M = (A * weights[:, None]).T @ B
        U, _, Vt = np.linalg.svd(M)
        Q_init = U @ Vt
        A_aligned = A @ Q_init
        sims = np.sum(A_aligned * B, axis=1)
        cutoff = np.percentile(sims, 15)
        mask = sims >= cutoff
        A_f = A[mask]
        B_f = B[mask]
        w_f = weights[mask]
        if len(A_f) < 50: return Q_init
        M_f = (A_f * w_f[:, None]).T @ B_f
        U, _, Vt = np.linalg.svd(M_f)
        return U @ Vt
    
    def _extract_features_std(self, words, Q, pivots1, pivots2):
        valid_indices = []
        valid_words = []
        for i, w in enumerate(words):
            if w in self.w2i_1900 and w in self.w2i_1990:
                valid_indices.append(i)
                valid_words.append(w)
        X_out = np.zeros((len(words), 11))
        if not valid_words: return X_out
        w_idx1 = [self.w2i_1900[w] for w in valid_words]
        w_idx2 = [self.w2i_1990[w] for w in valid_words]
        vecs1 = self.W1900[w_idx1]
        vecs2 = self.W1990[w_idx2]
        vecs1_aligned = vecs1 @ Q
        cos_sims = np.sum(vecs1_aligned * vecs2, axis=1)
        cos_dists = 1.0 - cos_sims
        prof1 = vecs1 @ pivots1.T
        prof2 = vecs2 @ pivots2.T
        prof1 = prof1 / (np.linalg.norm(prof1, axis=1, keepdims=True) + 1e-12)
        prof2 = prof2 / (np.linalg.norm(prof2, axis=1, keepdims=True) + 1e-12)
        pivot_sims = np.sum(prof1 * prof2, axis=1)
        pivot_dists = 1.0 - pivot_sims
        p1_mean = np.mean(prof1, axis=1, keepdims=True)
        p2_mean = np.mean(prof2, axis=1, keepdims=True)
        p1_c = prof1 - p1_mean
        p2_c = prof2 - p2_mean
        p1_std = np.linalg.norm(p1_c, axis=1) + 1e-12
        p2_std = np.linalg.norm(p2_c, axis=1) + 1e-12
        pearson = np.sum(p1_c * p2_c, axis=1) / (p1_std * p2_std)
        pearson_dist = 1.0 - pearson
        k_piv = 50
        top_p1 = np.argpartition(prof1, -k_piv, axis=1)[:, -k_piv:]
        top_p2 = np.argpartition(prof2, -k_piv, axis=1)[:, -k_piv:]
        pivot_jaccards = []
        for row1, row2 in zip(top_p1, top_p2):
            s1 = set(row1)
            s2 = set(row2)
            inter = len(s1 & s2)
            union = len(s1 | s2)
            pivot_jaccards.append(inter / union if union > 0 else 0)
        pivot_jaccards = np.array(pivot_jaccards)
        k_max = 100
        nn_1900_k100 = self._get_neighbors_batch(vecs1, self.W1900, k=k_max)
        nn_1990_k100 = self._get_neighbors_batch(vecs2, self.W1990, k=k_max)
        nn_1900 = nn_1900_k100
        nn_1990 = nn_1990_k100
        jac_k10 = []
        jac_k50 = []
        jac_k100 = []
        weighted_jaccards = []
        for i in range(len(valid_words)):
            s1_10 = set(nn_1900[i, -10:])
            s2_10 = set(nn_1990[i, -10:])
            inter10 = s1_10 & s2_10
            union10 = s1_10 | s2_10
            jac_k10.append(len(inter10) / len(union10) if union10 else 0)
            s1_50 = set(nn_1900[i])
            s2_50 = set(nn_1990[i])
            inter50 = s1_50 & s2_50
            union50 = s1_50 | s2_50
            jac_k50.append(len(inter50) / len(union50) if union50 else 0)
            s1_100 = set(nn_1900_k100[i])
            s2_100 = set(nn_1990_k100[i])
            inter100 = s1_100 & s2_100
            union100 = s1_100 | s2_100
            jac_k100.append(len(inter100) / len(union100) if union100 else 0)
            w_score = 0.0
            sum_w = 0.0
            for idx in union50:
                w = 1.0 / np.log(idx + 2)
                sum_w += w
                if idx in inter50:
                    w_score += w
            weighted_jaccards.append(w_score / sum_w if sum_w > 0 else 0.0)
        jac_k10 = np.array(jac_k10)
        jac_k50 = np.array(jac_k50)
        jac_k100 = np.array(jac_k100)
        weighted_jaccards = np.array(weighted_jaccards)
        k_context = 20
        nn_1900_c = nn_1900[:, -k_context:]
        context_shifts = []
        for i, neighbors in enumerate(nn_1900_c):
            t1 = vecs1[i]
            t2 = vecs2[i]
            sims1 = []
            sims2 = []
            for n_idx in neighbors:
                nw = self.i2w_1900[n_idx]
                if nw in self.w2i_1990:
                    v1 = self.W1900[n_idx]
                    v2 = self.W1990[self.w2i_1990[nw]]
                    sims1.append(t1 @ v1)
                    sims2.append(t2 @ v2)
            if len(sims1) < 5:
                context_shifts.append(1.0)
                continue
            shift = np.mean(np.abs(np.array(sims1) - np.array(sims2)))
            context_shifts.append(shift)
        context_shifts = np.array(context_shifts)
        rank1 = np.log([self.w2i_1900[w] + 1 for w in valid_words])
        rank2 = np.log([self.w2i_1990[w] + 1 for w in valid_words])
        rank_diff = np.abs(rank1 - rank2)
        X_fill = np.column_stack([
            cos_dists, pivot_dists, pearson_dist, pivot_jaccards, 
            jac_k10, jac_k50, jac_k100, weighted_jaccards, context_shifts,
            rank1, rank_diff
        ])
        X_out[valid_indices] = X_fill
        mask_oov = np.ones(len(words), dtype=bool)
        mask_oov[valid_indices] = False
        if np.any(mask_oov):
            X_out[mask_oov, 0] = 2.0
            X_out[mask_oov, 1] = 2.0 
            X_out[mask_oov, 2] = 2.0 
            X_out[mask_oov, 3] = 0.0 
            X_out[mask_oov, 4] = 0.0
            X_out[mask_oov, 5] = 0.0
            X_out[mask_oov, 6] = 0.0
            X_out[mask_oov, 7] = 0.0
            X_out[mask_oov, 8] = 1.0 
            X_out[mask_oov, 9] = np.log(100000)
            X_out[mask_oov, 10] = 0.0
        return X_out
    
    def _extract_features(self, words):
        return self._extract_features_std(words, self.Q, self.pivots_1900, self.pivots_1990)
    
    def _extract_features_with_Q(self, words, Q):
        valid_indices = []
        valid_words = []
        for i, w in enumerate(words):
            if w in self.w2i_1900 and w in self.w2i_1990:
                valid_indices.append(i)
                valid_words.append(w)
        X_out = np.zeros((len(words), 4))
        if not valid_words: return X_out
        w_idx1 = [self.w2i_1900[w] for w in valid_words]
        w_idx2 = [self.w2i_1990[w] for w in valid_words]
        vecs1 = self.W1900[w_idx1]
        vecs2 = self.W1990[w_idx2]
        vecs1_aligned = vecs1 @ Q
        cos_sims = np.sum(vecs1_aligned * vecs2, axis=1)
        cos_dists = 1.0 - cos_sims
        k = 50
        nn_1900 = self._get_neighbors_batch(vecs1, self.W1900, k=k)
        nn_1990 = self._get_neighbors_batch(vecs2, self.W1990, k=k)
        jaccards = []
        for n1_indices, n2_indices in zip(nn_1900, nn_1990):
            w1_set = {self.i2w_1900[idx] for idx in n1_indices}
            w2_set = {self.i2w_1990[idx] for idx in n2_indices}
            inter = len(w1_set & w2_set)
            union = len(w1_set | w2_set)
            jaccards.append(inter / union if union > 0 else 0.0)
        jaccards = np.array(jaccards)
        rank1 = np.log([self.w2i_1900[w] + 1 for w in valid_words])
        rank2 = np.log([self.w2i_1990[w] + 1 for w in valid_words])
        rank_diff = np.abs(rank1 - rank2)
        X_fill = np.column_stack([cos_dists, jaccards, rank1, rank_diff])
        X_out[valid_indices] = X_fill
        mask_oov = np.ones(len(words), dtype=bool)
        mask_oov[valid_indices] = False
        if np.any(mask_oov):
            X_out[mask_oov, 0] = 2.0
            X_out[mask_oov, 1] = 0.0
            X_out[mask_oov, 2] = 0.0
            X_out[mask_oov, 3] = np.log(100000)
            X_out[mask_oov, 4] = 0.0
        return X_out
    
    def _get_neighbors_batch(self, vectors, W, k=10):
        batch_size = 50
        n_samples = vectors.shape[0]
        neighbors = []
        for i in range(0, n_samples, batch_size):
            end = min(i + batch_size, n_samples)
            v_batch = vectors[i:end]
            sims = v_batch @ W.T 
            partition = np.argpartition(sims, -k, axis=1)[:, -k:]
            batch_sims_k = np.take_along_axis(sims, partition, axis=1)
            sorted_idx_within_k = np.argsort(batch_sims_k, axis=1)
            sorted_partition = np.take_along_axis(partition, sorted_idx_within_k, axis=1)
            neighbors.append(sorted_partition)
        return np.concatenate(neighbors, axis=0)
    
    def predict_change(self, words):
        X = self._extract_features(words)
        X_scaled = self.scaler.transform(X)
        return self.clf.predict(X_scaled)

MODEL = SemanticChangeModel(W1900, W1990, w2i_1900, w2i_1990)
MODEL.fit(train_df)


Start...
Obliczono alignment
Nauczono za pomocą VotingEnsemble


## Ewaluacja

Uruchomienie poniższej komórki pozwoli sprawdzić, ile punktów zdobyłoby Twoje rozwiązanie na danych walidacyjnych.

Upewnij się przed wysłaniem, że cały notebook wykonuje się od początku do końca bez błędów i bez ingerencji użytkownika po wykonaniu polecenia `Run All`.

In [9]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################
def compute_score(bal_acc: float) -> float:
    """
    Oblicza wynik punktowy na podstawie wartości zbalansowanej dokładności.

    :param bal_acc: Wartość float w zakresie [0.0, 1.0]
    :return: Wynik punktowy zgodny z określoną funkcją
    """
    if bal_acc <= 0.7:
        return 0
    elif 0.7 < bal_acc < 0.87:
        return int(round(100 * (bal_acc - 0.7) / (0.87 - 0.7)))
    else:
        return 100


def evaluate_algorithm(dataset_df, model, verbose=False):
    """
    Ewaluacja modelu wykrywania zmiany znaczenia słów na podanym zbiorze danych.

    Parametry
    ----------
    dataset_df : pd.DataFrame
        Oznaczony zbiór danych z kolumnami:
          - 'word'  : słowo (string)
          - 'label' : etykieta 0 = stabilne, 1 = zmienione

    model : obiekt
        Obiekt posiadający metodę:
            predict_change(words: list[str]) -> list[int] {0,1}

    verbose : bool
        Jeśli True, wypisuje dodatkowe informacje.

    Zwraca
    -------
    points : float
        Wynik punktowy oparty na zbalansowanej dokładności.
    """

    # Wyodrębnij słowa i etykiety z datasetu
    words = dataset_df["word"].astype(str).tolist()
    ys = dataset_df["label"].astype(int).tolist()

    # Pobierz predykcje dla całej listy słów
    preds = model.predict_change(words)

    # Konwersja predykcji i etykiet na tablice numpy
    preds = np.array(preds, dtype=np.int32)
    ys = np.array(ys, dtype=np.int32)

    # Zbalansowana dokładność
    bal_acc = balanced_accuracy_score(ys, preds)

    # Konwersja dokładności na punkty konkursowe
    points = compute_score(bal_acc)

    if verbose:
        print(f"\nLiczba próbek: {len(dataset_df)}")
        print(f"Balanced accuracy: {bal_acc:.4f}")
        print(f"Wynik punktowy: {points}")

    return points


if not FINAL_EVALUATION_MODE:
    _ = evaluate_algorithm(valid_df, MODEL, verbose=True)


Liczba próbek: 832
Balanced accuracy: 0.8741
Wynik punktowy: 100
